# VisionTracker — Kaggle Remote Server

**GPU:** P100 (16 GB VRAM) — recommended over T4 for larger context

**Model:** LLaVA-1.6-Mistral-7B-Q4_K_M (~5 GB total)

**Steps:**
1. Enable GPU: Settings → Accelerator → P100
2. Add your `NGROK_AUTHTOKEN` secret (Settings → Secrets)
3. Run all cells
4. Copy the public URL, then on your machine:
   `python main.py --remote-url https://xxx.ngrok.io`


In [ ]:
# ── Cell 1: Install dependencies ─────────────────────────────────────────
import subprocess, sys

# llama-cpp-python: prebuilt CUDA 12.1 wheels (no compile needed)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'llama-cpp-python',
    '--extra-index-url', 'https://abetlen.github.io/llama-cpp-python/whl/cu121'
], check=True)

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'fastapi', 'uvicorn[standard]', 'python-multipart',
    'pyngrok', 'pillow', 'pydantic',
    'transformers', 'accelerate', 'torch'
], check=True)

print('✅ Dependencies installed')


In [ ]:
# ── Cell 2: Configure ngrok ───────────────────────────────────────────────
from kaggle_secrets import UserSecretsClient
from pyngrok import ngrok

try:
    token = UserSecretsClient().get_secret('NGROK_AUTHTOKEN')
    print('✅ Loaded NGROK_AUTHTOKEN from Kaggle secrets')
except Exception:
    token = input('Enter ngrok authtoken (https://dashboard.ngrok.com): ').strip()

if not token:
    raise ValueError('ngrok authtoken required')

ngrok.set_auth_token(token)
print('✅ ngrok configured')


In [ ]:
# ── Cell 3: Download LLaVA-1.6-Mistral-7B (best indoor VLM, <10 GB) ──────
# Main GGUF:   llava-1.6-mistral-7b.Q4_K_M.gguf  (~4.4 GB)
# Vision enc:  mmproj-model-f16.gguf              (~631 MB)
# HF repo:     https://huggingface.co/cjpais/llava-1.6-mistral-7b-gguf
import os

BASE = 'https://huggingface.co/cjpais/llava-1.6-mistral-7b-gguf/resolve/main'
MODEL_PATH  = '/kaggle/working/llava-1.6-mistral-7b.Q4_K_M.gguf'
MMPROJ_PATH = '/kaggle/working/mmproj-model-f16.gguf'

def _download(url, dest):
    if os.path.exists(dest):
        print(f'✅ Exists: {dest}')
        return
    print(f'⬇️  Downloading {os.path.basename(dest)}...')
    os.system(f'wget -q --show-progress "{url}" -O "{dest}"')
    print(f'✅ Done: {dest}')

_download(f'{BASE}/llava-1.6-mistral-7b.Q4_K_M.gguf', MODEL_PATH)
_download(f'{BASE}/mmproj-model-f16.gguf', MMPROJ_PATH)

total = (os.path.getsize(MODEL_PATH) + os.path.getsize(MMPROJ_PATH)) / 1e9
print(f'\n📦 Total: {total:.2f} GB')


In [ ]:
# ── Cell 4: Copy server.py ────────────────────────────────────────────────
# Option A: upload server.py via Kaggle Add Data → upload
# Option B: paste from GitHub raw URL below
import os
SERVER_SRC = '/kaggle/input/visiontracker-server/server.py'  # if added as dataset
SERVER_DST = '/kaggle/working/server.py'

if os.path.exists(SERVER_SRC):
    import shutil; shutil.copy(SERVER_SRC, SERVER_DST)
    print(f'✅ Copied from dataset: {SERVER_DST}')
else:
    os.system('wget -q https://raw.githubusercontent.com/your-repo/visiontracker/main/remote_server/server.py -O /kaggle/working/server.py')
    print('✅ Downloaded server.py')


In [ ]:
# ── Cell 5: Start server ──────────────────────────────────────────────────
import subprocess, threading, time, nest_asyncio
nest_asyncio.apply()

public_url = ngrok.connect(8000, 'http')

print('━' * 62)
print('🚀  VisionTracker Remote Server is READY')
print('━' * 62)
print(f'📡  Public URL : {public_url}')
print(f'    Health     : {public_url}/health')
print(f'    Identify   : {public_url}/identify')
print('━' * 62)
print('💡  Client command:')
print(f'    python main.py --remote-url {public_url}')
print('━' * 62)

def _run():
    cmd = [
        'python', '/kaggle/working/server.py',
        '--model-path',  MODEL_PATH,
        '--mmproj-path', MMPROJ_PATH,
        '--host', '0.0.0.0', '--port', '8000',
    ]
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end='')

threading.Thread(target=_run, daemon=True).start()
print('\nKeep this notebook running!')
try:
    while True: time.sleep(1)
except KeyboardInterrupt:
    ngrok.disconnect(public_url); print('Stopped.')
